In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# Crear widgets
dbutils.widgets.text("catalogo", "catalog_cr")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
# Obtener valores de los widgets
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# Crear Dataframe a partir de tabla "crime_reports_transformed".
df_crime_reports_transformed = spark.table(f"{catalogo}.{esquema_source}.crime_reports_transformed")
df_crime_reports_transformed.display()

In [0]:
# Contar filas de df_crime_reports_transformed
df_crime_reports_transformed.count()

In [0]:
# Filtrar por tipo de crimen "Extorsion"
df_extortion_reports = df_crime_reports_transformed.filter(col("crime_name")=="Extorsión")
df_extortion_reports.display()

In [0]:
# Agrupar por ubigeo: total, min, max, año del min, año del max y promedio de total_n_crimes
df_extortion_reports_grouped = df_extortion_reports.groupBy("ubigeo","department","province","district").agg(
    F.sum("total_n_crimes").alias("total_crimes"),
    F.min("total_n_crimes").alias("min_crimes"),
    F.max("total_n_crimes").alias("max_crimes"),
    F.min(F.struct("total_n_crimes", "report_year")).getField("report_year").alias("year_min_crimes"),
    F.max(F.struct("total_n_crimes", "report_year")).getField("report_year").alias("year_max_crimes"),
    F.round(F.avg("total_n_crimes"), 2).alias("avg_crimes")
)
df_extortion_reports_grouped.display()

In [0]:
# Añadir columna id_report a df_transformed desde el n°1
df_extortion_reports_grouped = df_extortion_reports_grouped.withColumn("report_id", monotonically_increasing_id() + 1)
df_extortion_reports_grouped.display()

In [0]:
# Contar filas df_extortion_reports_grouped
df_extortion_reports_grouped.count()

In [0]:
# Seleccionar columnas específicas, con report_id al inicio.
df_extortion_reports_reordered = df_extortion_reports_grouped.select(
    col("report_id"),
    col("ubigeo"),
    col("department"),
    col("province"),
    col("district"),
    col("total_crimes").alias("total_extortions"),
    col("min_crimes").alias("min_extortions"),
    col("max_crimes").alias("max_extortions"),
    col("year_min_crimes").alias("year_min_extortions"),
    col("year_max_crimes").alias("year_max_extortions"),
    col("avg_crimes").alias("avg_extortions")
)
df_extortion_reports_reordered.display()

In [0]:
# Contar filas de df_extortion_reports_reordered
df_extortion_reports_reordered.count()

In [0]:
# Ingestar data en tabla "extortion_reports"
df_extortion_reports_reordered.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.extortion_reports")